# 順序分類GPとActive Learning：初学者向け実践ガイド

このNotebookでは、製品品質などの **順序がある3クラス（0: 不良・1: 要確認・2: 良好）** を対象に、
**学習 → 予測確率の解釈 → Active Learningで候補提案 → ラベルの取得 → 再学習** を実行します。
最後にカテゴリ入力を持つMixed Ordinal GPと、入力制約付きの候補探索も試します。

通常のマルチクラスGPと違い、Ordinal GPは**1つの潜在関数と単調な閾値**で順序を表します。
「クラス0と1の差」と「クラス1と2の差」が等しいとは仮定しません。

関連資料：[理論（数式の導出）](../../docs/theory/24_classification.md) /
[実装との対応](../../docs/guide/24_classification.md) /
[分類モデル一覧](../../docs/models/classification.md)


## 0. 全体の流れ

1. 初期の観測データ `X, y`（ラベルは0、1、2）を用意する
2. `OrdinalSingleTaskGPClassifier` を学習する
3. `posterior(X)` と `predict_proba(X)` の違いを確認する
4. Entropy・BALDで「次にラベルを測りたい条件」を評価する
5. q=2の候補をまとめて提案する
6. Mixed（連続＋カテゴリ）入力も試す
7. 未観測候補から選択→観測→再学習を**2回**繰り返す
8. 保留データで評価指標を確認する

**重要：** 実験条件を提案するだけではラベルは得られません。
実際には検査・実験・専門家判定でラベルを取得します。
ここでは理解のために**人工的なラベル生成規則**を検査装置の代わりに使います。
実データと比較するための性能実証ではありません。


## 1. 初期データ：入力 `X` と順序ラベル `y`

2つの連続入力条件を使います。第一列は品質に主な影響を与える条件、
第二列は補助的な条件と想定します。
説明のため、隠れた品質スコア `x0 + 0.12*x1` を
0.4と0.75で区切って3つの順序ラベルを作ります。

**ラベルは観測値としてモデルへ渡します。**
実際のモデルが、この人工規則や境界値0.4・0.75を知っているわけではありません。


In [ ]:
import torch

from robotorchan.acquisition.active_learning.classification import (
    BALD,
    MarginUncertainty,
    PredictiveEntropy,
    qBALD,
    qPredictiveEntropy,
)
from robotorchan.models.classification import (
    MixedOrdinalSingleTaskGPClassifier,
    OrdinalSingleTaskGPClassifier,
)
from robotorchan.optim.backends.sampling import optimize_acqf_sampling
from robotorchan.optim.constraints.contracts import CandidateConstraints
from robotorchan.optim.domains.variable_space import MixedVariableSpace
from robotorchan.optim.strategies.sobol import SobolSearchStrategy

torch.manual_seed(42)
dtype = torch.double
thresholds_for_synthetic_labels = torch.tensor([0.4, 0.75], dtype=dtype)

condition = torch.linspace(0.05, 0.95, 18, dtype=dtype)
auxiliary = 0.5 + 0.35 * torch.sin(6.0 * condition)
train_X = torch.stack((condition, auxiliary), dim=-1)
synthetic_quality = train_X[:, 0] + 0.12 * train_X[:, 1]
train_Y = torch.bucketize(
    synthetic_quality.contiguous(), thresholds_for_synthetic_labels
)
print("入力形状:", train_X.shape, "ラベル形状:", train_Y.shape)
print("実測扱いのクラス:", train_Y.unique(sorted=True))
assert train_X.shape == (18, 2)
assert torch.equal(train_Y.unique(), torch.arange(3))


## 2. 単一潜在GPで順序ラベルを学習する

`num_classes=3` は順序ラベル0、1、2の3段階を意味します。
`inducing_points=8` は変分GPの計算を近似する代表点の数であり、
追加の観測データ8点を意味しません。

`fit()` はGPカーネル・変分分布と、**Ordered-probitの閾値**を学習します。
このNotebookではCIで動かしやすいように学習回数を小さくしています。
実務では検証データで学習回数・学習率・誘導点数を調整してください。


In [ ]:
model = OrdinalSingleTaskGPClassifier(
    train_X, train_Y, num_classes=3, inducing_points=8
)
model.fit(epochs=4, lr=0.03)
learned_thresholds = model.likelihood.thresholds.detach()
print("学習した閾値:", learned_thresholds)
print("クラスの並び:", model.class_labels)
assert learned_thresholds.shape == (2,)
assert bool((learned_thresholds[1:] > learned_thresholds[:-1]).all())


## 3. 潜在値・閾値・予測確率の違い

Ordinal GPは**クラス数個の潜在GP**を作りません。
一つの潜在GPから得た連続値 `f(x)` と、順序を保つ閾値を用います。

- `posterior(X)`：潜在関数のGaussianな事後分布。平均の最終次元は**1**
- `predict_proba(X)`：各クラスへの**事後予測確率**。最終次元は**3**
- `predict_class(X)`：確率が最大のクラス番号
- `predictive_entropy(X)`：予測ラベルがどれほど曖昧かを表す値

閾値を潜在平均に適用するだけでなく、
**潜在GPの分散を含めて積分**することで予測確率を計算します。
詳細は[理論第24章](../../docs/theory/24_classification.md)にあります。


In [ ]:
test_X = torch.tensor(
    [[0.2, 0.5], [0.5, 0.5], [0.8, 0.5]], dtype=dtype
)
latent = model.posterior(test_X)
probabilities = model.predict_proba(test_X)
predicted_classes = model.predict_class(test_X)
entropy = model.predictive_entropy(test_X)

print("潜在平均（確率ではない）:", latent.mean.squeeze(-1))
print("クラス0・1・2の予測確率:\n", probabilities)
print("予測ラベル:", predicted_classes)
print("ラベルのEntropy:", entropy)
assert latent.mean.shape == (3, 1)
assert probabilities.shape == (3, 3)
torch.testing.assert_close(probabilities.sum(-1), torch.ones(3, dtype=dtype))


### 条件付きクラス確率と平均予測確率は違う

`sample_class_probabilities` は潜在GPから複数の値をサンプルし、
それぞれをOrdered-probitの条件付き確率へ変換します。
その**サンプル間のばらつき**がモデル由来の不確実性を表します。
一方 `predict_proba` は潜在Gaussian分布を解析的に周辺化した
**予測ラベルの確率**です。

両者が似た値になっても、前者を単純に複製したものが後者ではありません。


In [ ]:
probability_samples = model.sample_class_probabilities(
    test_X, sample_shape=torch.Size([32])
)
print("確率サンプルの形状:", probability_samples.shape)
print("最初の入力に対する確率サンプルの平均:", probability_samples[:, 0].mean(0))
print("解析的な予測確率:", probabilities[0])
assert probability_samples.shape == (32, 3, 3)


## 4. Active Learning：次に測る条件を比較する

未測定の候補を並べ、獲得関数で点数を付けます。

- **PredictiveEntropy**：ラベルが曖昧な条件を選ぶ
- **MarginUncertainty**：上位2クラスの予測確率差が小さい条件を選ぶ
- **BALD**：追加観測で潜在モデルについて情報が増えそうな条件を選ぶ

`BALD` は不確実性をモンテカルロサンプルから近似するので、
乱数によって点数が多少変わります。
**スコアが大きいことは、品質が高いことを意味しません。**


In [ ]:
candidate_x0 = torch.linspace(0.05, 0.95, 12, dtype=dtype)
candidate_X = torch.stack(
    (candidate_x0, torch.full_like(candidate_x0, 0.5)), dim=-1
)
# 単一点の獲得関数には [候補数, q=1, 入力次元] を渡します。
candidate_batches = candidate_X.unsqueeze(-2)
entropy_scores = PredictiveEntropy(model)(candidate_batches)
margin_scores = MarginUncertainty(model)(candidate_batches)
bald_scores = BALD(model, num_samples=16)(candidate_batches)
print("Entropy候補:", candidate_X[entropy_scores.argmax()])
print("Margin候補:", candidate_X[margin_scores.argmax()])
print("BALD候補:", candidate_X[bald_scores.argmax()])
assert entropy_scores.shape == margin_scores.shape == bald_scores.shape == (12,)
assert torch.isfinite(bald_scores).all()


## 5. q=2：2つの条件をまとめて提案する

`qPredictiveEntropy` は2候補の周辺Entropyを平均します。
**候補間の重複情報を直接抑制する関数ではありません。**

`qBALD` は潜在GPの同時サンプルから候補間の依存を考慮します。
ただしラベルベクトルの組合せ数が `C**q <= 256` である必要があります。
この例は3クラス・2候補なので `3**2=9` 通りです。

`SobolSearchStrategy` は既存の候補生成器で、**獲得関数が高い条件**を提案します。
提案結果にはクラスラベルではなく入力条件が入ります。


In [ ]:
bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]], dtype=dtype)
strategy = SobolSearchStrategy(bounds, num_samples=16, seed=17)
result = strategy.optimize(qPredictiveEntropy(model), q=2)
print("提案された2つの連続条件:\n", result.candidates)
print("周辺Entropy平均:", result.acquisition_value)
joint_bald_score = qBALD(model, num_samples=16)(
    result.candidates.unsqueeze(0)
)
print("同じ候補のjoint qBALD:", joint_bald_score)
assert result.candidates.shape == (2, 2)
assert joint_bald_score.shape == (1,)
assert torch.isfinite(joint_bald_score).all()


## 6. Mixed Ordinal GP：入力に設備カテゴリを含む場合

**出力のクラス0・1・2は順序付き**ですが、
**入力の設備カテゴリ0・1・2には順序がない**とします。

`cat_dims=[1]` は入力の2列目（設備ID）に
カテゴリ用カーネルを使う指定です。
`MixedOrdinalSingleTaskGPClassifier` は、
通常のOrdinal GPと同じOrdered-probit尤度・確率予測の仕組みを再利用します。


In [ ]:
mixed_X = train_X.clone()
mixed_X[:, 1] = torch.arange(mixed_X.shape[0]) % 3
mixed_quality = mixed_X[:, 0] + 0.08 * (mixed_X[:, 1] == 2).to(dtype)
mixed_Y = torch.bucketize(
    mixed_quality.contiguous(), thresholds_for_synthetic_labels
)
mixed_model = MixedOrdinalSingleTaskGPClassifier(
    mixed_X, mixed_Y, num_classes=3, cat_dims=[1], inducing_points=8
)
mixed_model.fit(epochs=4, lr=0.03)
valid_mixed_X = torch.tensor(
    [[0.2, 0.0], [0.5, 1.0], [0.8, 2.0]], dtype=dtype
)
mixed_probabilities = mixed_model.predict_proba(valid_mixed_X)
print("Mixed入力のクラス確率:\n", mixed_probabilities)
assert mixed_probabilities.shape == (3, 3)
torch.testing.assert_close(
    mixed_probabilities.sum(-1), torch.ones(3, dtype=dtype)
)


### Mixed候補の提案：合法なカテゴリと入力制約を指定する

モデル側の `cat_dims` だけでは候補最適化器はカテゴリ値を知りません。
`MixedVariableSpace` に **合法な設備ID 0・1・2** を明示します。

ここでは連続条件（第1列）が0.25以上という**既知の入力制約**も付けます。
これは「良好クラスの確率が0.25以上」などの**モデル出力制約ではありません**。


In [ ]:
mixed_bounds = torch.tensor(
    [[0.0, 0.0], [1.0, 2.0]], dtype=dtype
)
mixed_space = MixedVariableSpace(
    mixed_bounds, categorical_values={1: (0.0, 1.0, 2.0)}
)
input_constraints = CandidateConstraints(
    inequality_constraints=(
        (torch.tensor([0]), torch.tensor([1.0], dtype=dtype), 0.25),
    )
)
mixed_candidates, mixed_acq_value = optimize_acqf_sampling(
    qPredictiveEntropy(mixed_model),
    mixed_bounds,
    q=2,
    num_samples=24,
    method="sobol",
    seed=7,
    variable_space=mixed_space,
    constraints=input_constraints,
)
print("制約を満たすMixed候補:\n", mixed_candidates)
print("獲得関数値:", mixed_acq_value)
assert mixed_candidates.shape == (2, 2)
assert (mixed_candidates[:, 0] >= 0.25).all()
assert torch.isin(
    mixed_candidates[:, 1], torch.tensor([0.0, 1.0, 2.0])
).all()


## 7. 次の実験を提案し、結果を受け取って再学習する（2ラウンド）

ここからが実際のActive Learningのサイクルです。

1. **Ask**：未観測候補だけでEntropyを計算し、測定条件を提案する
2. **Tell**：**提案された後に**初めて検査して順序ラベルを得る
3. 観測済みデータへ追加し、選んだ候補をプールから除く
4. 新しいデータでモデルを学習し直して、次の条件を提案する

実務では `torch.bucketize(...)` に相当する部分を
実際の検査結果で置き換えます。ここでは人工ラベル生成規則を使います。
未観測プールの正解ラベルは**候補選択に利用しません**。


In [ ]:
unobserved_x0 = torch.linspace(0.08, 0.92, 10, dtype=dtype)
unobserved_pool = torch.stack(
    (unobserved_x0, torch.full_like(unobserved_x0, 0.5)), dim=-1
)
observed_X = train_X.clone()
observed_Y = train_Y.clone()

for round_number in range(1, 3):
    current_model = OrdinalSingleTaskGPClassifier(
        observed_X, observed_Y, num_classes=3, inducing_points=8
    )
    current_model.fit(epochs=3, lr=0.03)
    scores = PredictiveEntropy(current_model)(
        unobserved_pool.unsqueeze(-2)
    )
    selected_index = int(scores.argmax().item())
    suggested_X = unobserved_pool[selected_index : selected_index + 1]

    # ラベルは「提案後に実験で観測する」ことを人工規則で模擬します。
    observed_quality = suggested_X[:, 0] + 0.12 * suggested_X[:, 1]
    newly_observed_Y = torch.bucketize(
        observed_quality.contiguous(), thresholds_for_synthetic_labels
    )
    print(
        f"Round {round_number}: 提案={suggested_X.tolist()}, "
        f"観測ラベル={newly_observed_Y.item()}"
    )

    observed_X = torch.cat((observed_X, suggested_X), dim=0)
    observed_Y = torch.cat((observed_Y, newly_observed_Y), dim=0)
    keep = torch.ones(len(unobserved_pool), dtype=torch.bool)
    keep[selected_index] = False
    unobserved_pool = unobserved_pool[keep]

assert observed_X.shape[0] == train_X.shape[0] + 2
assert len(unobserved_pool) == 8

# 2ラウンド目で取得したラベルも反映し、全観測データで学習し直します。
final_model = OrdinalSingleTaskGPClassifier(
    observed_X, observed_Y, num_classes=3, inducing_points=8
)
final_model.fit(epochs=3, lr=0.03)


## 8. 保留データで予測の読み方を確認する

実務では学習や候補選択に使わなかった保留データで検証します。
ここでは別の人工点を使い、**Accuracy・NLL・Brier score** の計算を確認します。
以下の値は少数epochの簡易例の結果であり、実用精度を示すものではありません。

- **Accuracy**：最大確率クラスが正しい比率
- **NLL**：正解クラスに割り当てた確率を評価（小さいほどよい）
- **Brier score**：確率ベクトルとOne-hot正解の二乗誤差（小さいほどよい）
- **順序の誤差**：必要ならクラス番号の絶対差も評価できますが、
  番号間隔が同じ品質差を意味するとは限りません。


In [ ]:
holdout_x0 = torch.linspace(0.1, 0.9, 12, dtype=dtype)
holdout_X = torch.stack(
    (holdout_x0, torch.full_like(holdout_x0, 0.5)), dim=-1
)
holdout_quality = holdout_X[:, 0] + 0.12 * holdout_X[:, 1]
holdout_Y = torch.bucketize(
    holdout_quality.contiguous(), thresholds_for_synthetic_labels
)
# 最終ラウンドの学習器で保留点を評価します。選択には利用しません。
holdout_proba = final_model.predict_proba(holdout_X)
accuracy = (holdout_proba.argmax(-1) == holdout_Y).double().mean()
correct_probability = holdout_proba.gather(
    -1, holdout_Y.unsqueeze(-1)
).squeeze(-1)
nll = -correct_probability.clamp_min(1e-12).log().mean()
one_hot = torch.nn.functional.one_hot(holdout_Y, num_classes=3).to(dtype)
brier = ((holdout_proba - one_hot) ** 2).sum(dim=-1).mean()
print("Accuracy:", float(accuracy))
print("NLL:", float(nll))
print("Multiclass Brier:", float(brier))
assert torch.isfinite(nll)
assert torch.isfinite(brier)


## 9. 実務での確認事項と次の学習

- **出力の順序**：0、1、2の意味と方向を先に定義し、全データで揃える
- **クラス不均衡**：3つのラベルが十分に観測されているか確認する
- **確率の校正**：AccuracyだけでなくNLL、Brier score、クラス別Recallを確認する
- **提案品質**：同じラベル予算でランダムサンプリングとも比較する
- **入力制約**：設備ID・可行領域など、実験可能な条件を探索器へ指定する
- **q-batch**：周辺平均スコアとjoint qBALDの違い、`C**q <= 256` を確認する
- **過信しない**：小さな人工例の学習成功は実際の汎化性能・安全性の保証ではない

**まとめ：** Ordinal GPは、順序ラベルを単なる数値回帰として扱うことなく、
単一の潜在GPとOrdered-probit閾値で確率をモデル化します。
学習・予測・単一点Active Learning・q-batch・Mixed入力・候補生成を
公開APIの組合せで実行できます。

次に読む：[Ordinal GPの理論](../../docs/theory/24_classification.md) /
[実装ガイド](../../docs/guide/24_classification.md) /
[分類モデルと対応範囲](../../docs/models/classification.md)
